In [1]:
#!/usr/bin/env python3
import os
import sys
import numpy as np

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()
CRPROPA_BUILD = BASE_DIR
OUTDIR = os.environ.get(
    "CRPROPA_TEST_OUTDIR",
    os.path.join(BASE_DIR, "velocity_test_outputs"),
)
NEVENTS = 1

sys.path.append(CRPROPA_BUILD)
from crpropa import *

os.makedirs(OUTDIR, exist_ok=True)

# Energías cinéticas del electrón [eV]
energies_eV = [1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9, 1e12]

dSrc = 1.0 * Mpc

def enable_if_exists(out, colname):
    if hasattr(Output, colname):
        out.enable(getattr(Output, colname))
    else:
        print(f"[WARNING] Output.{colname} not found")

def configure_output(out):
    out.disableAll()

    for col in [
        "TrajectoryLengthColumn",
        "TimeColumn",
        "CurrentIdColumn",
        "CurrentEnergyColumn",
        "CurrentPositionColumn",
        "CurrentDirectionColumn",
    ]:
        enable_if_exists(out, col)

    out.setEnergyScale(eV)
    out.setLengthScale(Mpc)

for E_eV in energies_eV:
    tag = f"E{E_eV:.0e}"

    source = Source()
    source.add(SourcePosition(Vector3d(0, 0, 0)))
    source.add(SourceDirection(Vector3d(1, 0, 0)))
    source.add(SourceParticleType(11))  # electron
    source.add(SourceEnergy(E_eV * eV))

    event_file = os.path.join(OUTDIR, f"event_{tag}.txt")
    traj_file = os.path.join(OUTDIR, f"traj_{tag}.txt")

    event_out = TextOutput(event_file, Output.Event3D)
    traj_out = TextOutput(traj_file, Output.Trajectory3D)

    configure_output(event_out)
    configure_output(traj_out)

    observer = Observer()
    observer.add(ObserverSurface(Sphere(Vector3d(dSrc, 0, 0), 1 * kpc)))
    observer.onDetection(event_out)

    sim = ModuleList()

    # Sin interacciones, sin campo magnético, sin redshift.
    sim.add(SimplePropagation(1 * pc, 10 * kpc))
    sim.add(traj_out)
    sim.add(observer)
    sim.add(MaximumTrajectoryLength(2 * dSrc))

    sim.setShowProgress(False)

    print(f"Running velocity test: Ekin = {E_eV:.3e} eV")
    sim.run(source, NEVENTS, True)

    event_out.close()
    traj_out.close()

print(f"Done. Outputs in {OUTDIR}")

Running velocity test: Ekin = 1.000e+03 eV
crpropa::ModuleList: Number of Threads: 8
Running velocity test: Ekin = 1.000e+04 eV
crpropa::ModuleList: Number of Threads: 8
Running velocity test: Ekin = 1.000e+05 eV
crpropa::ModuleList: Number of Threads: 8
Running velocity test: Ekin = 1.000e+06 eV
crpropa::ModuleList: Number of Threads: 8
Running velocity test: Ekin = 1.000e+07 eV
crpropa::ModuleList: Number of Threads: 8
Running velocity test: Ekin = 1.000e+08 eV
crpropa::ModuleList: Number of Threads: 8
Running velocity test: Ekin = 1.000e+09 eV
Running velocity test: Ekin = 1.000e+12 eV
crpropa::ModuleList: Number of Threads: 8
crpropa::ModuleList: Number of Threads: 8
Done. Outputs in /home/ciemat-t13/CRPropa_bundle/examples/velocity_test_outputs


In [4]:
#!/usr/bin/env python3
import os
import glob
import numpy as np

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

OUTDIR = os.environ.get(
    "CRPROPA_TEST_OUTDIR",
    os.path.join("/tmp", "bfield_larmor_outputs"),
)
B_NG = 1.0

eV_J = 1.602176634e-19
c_SI = 2.99792458e8
m_e_kg = 9.1093837015e-31
q_e_C = 1.602176634e-19
kpc_m = 3.0856775814913673e19

B_T = B_NG * 1e-13

def larmor_radius_kpc(Ekin_eV):
    T = Ekin_eV * eV_J
    mc2 = m_e_kg * c_SI**2

    # pc = sqrt(T^2 + 2 T mc^2)
    pc = np.sqrt(T * (T + 2.0 * mc2))
    p = pc / c_SI

    r_m = p / (q_e_C * B_T)
    return r_m / kpc_m

def fit_circle(x, y):
    M = np.column_stack([x, y, np.ones_like(x)])
    rhs = -(x**2 + y**2)

    A, B, C = np.linalg.lstsq(M, rhs, rcond=None)[0]

    xc = -A / 2.0
    yc = -B / 2.0
    r = np.sqrt((A**2 + B**2) / 4.0 - C)

    return xc, yc, r

summary = []

for fname in sorted(glob.glob(os.path.join(OUTDIR, "traj_larmor_E*.txt"))):
    data = np.loadtxt(fname, comments="#")

    if data.ndim == 1:
        data = data.reshape(1, -1)

    base = os.path.basename(fname)
    E_eV = float(base.split("traj_larmor_E")[1].split(".txt")[0])

    # Columnas esperadas:
    # D, time, ID, E, X, Y, Z, Px, Py, Pz
    x = data[:, 4]
    y = data[:, 5]

    if len(x) < 10:
        print(f"Not enough points in {fname}")
        continue

    xc, yc, r_fit = fit_circle(x, y)
    r_th = larmor_radius_kpc(E_eV)

    rel_err = abs(r_fit - r_th) / r_th

    summary.append([E_eV, r_th, r_fit, rel_err, xc, yc])

files = sorted(glob.glob(os.path.join(OUTDIR, "traj_larmor_E*.txt")))

print("OUTDIR =", OUTDIR)
print("Files found =", len(files))
for f in files[:10]:
    print(f)

if len(summary) == 0:
    raise RuntimeError(
        f"No valid trajectory files found in {OUTDIR}. "
        "Check CRPROPA_TEST_OUTDIR and the filename pattern traj_larmor_E*.txt"
    )

summary = np.array(summary)
summary = summary[np.argsort(summary[:, 0])]

np.savetxt(
    os.path.join(OUTDIR, "larmor_summary.txt"),
    summary,
    header="Ekin_eV r_theory_kpc r_fit_kpc relative_error x_center_kpc y_center_kpc"
)

print("E_eV        r_theory[kpc]   r_fit[kpc]      rel_error")
for row in summary:
    print(f"{row[0]:.3e}  {row[1]:.6e}     {row[2]:.6e}   {row[3]:.6e}")

if plt is not None:
    plt.figure(figsize=(6, 4))
    plt.loglog(summary[:, 0], summary[:, 1], marker="o", label="Theory")
    plt.loglog(summary[:, 0], summary[:, 2], marker="s", ls="--", label="CRPropa fit")
    plt.xlabel("Kinetic energy [eV]")
    plt.ylabel(r"$r_L$ [kpc]")
    plt.legend()
    #plt.grid(True, which="both")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, "larmor_radius_comparison.pdf"))
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.semilogx(summary[:, 0], summary[:, 3], marker="o")
    plt.xlabel("Kinetic energy [eV]")
    plt.ylabel("Relative error")
    #plt.grid(True, which="both")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, "larmor_relative_error.pdf"))
    plt.show()


OUTDIR = /tmp/bfield_larmor_outputs
Files found = 0


RuntimeError: No valid trajectory files found in /tmp/bfield_larmor_outputs. Check CRPROPA_TEST_OUTDIR and the filename pattern traj_larmor_E*.txt